# EXEMPLO 1: Conversão Mínima com Docling
## Aula 5 — Referência Rápida
### MBA em RAG & CAG Aplicados a Direito e Segurança Pública

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**⏱️ Duração:** 15 minutos | **Tipo:** Referência Rápida  
Este exemplo mostra o mínimo necessário para converter um documento com Docling
e gerar chunks prontos para indexação. Use como template de referência.

In [ ]:
!pip install docling>=2.0.0 reportlab --quiet
print("✅ Pronto")

In [ ]:
# ─── Template mínimo: Docling → Markdown → Chunks ─────────────────────────────
import warnings, json
from pathlib import Path
warnings.filterwarnings('ignore')

from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker

# ─── 1. Criar PDF de exemplo
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm

CAMINHO_PDF = '/tmp/exemplo_docling.pdf'
doc_rl = SimpleDocTemplate(CAMINHO_PDF, pagesize=A4,
    rightMargin=2*cm, leftMargin=2*cm, topMargin=2*cm, bottomMargin=2*cm)
s = getSampleStyleSheet()
doc_rl.build([
    Paragraph('Súmula 440 - STJ', s['Heading1']),
    Paragraph('Fixada a pena-base no mínimo legal, é vedado o estabelecimento '
              'do regime prisional mais gravoso do que o cabível em razão da '
              'sanção imposta, com base apenas na gravidade abstrata do delito.', s['Normal']),
    Paragraph('Contexto Jurídico', s['Heading2']),
    Paragraph('A Súmula 440/STJ consolida entendimento de que a fundamentação '
              'da pena exige elementos concretos individualizados, vedando a '
              'referência genérica à periculosidade ou gravidade abstrata. '
              'Aplica-se em especial ao estabelecimento de regime prisional '
              'e à valoração das circunstâncias judiciais (art. 59 CP).', s['Normal']),
])
print(f"✅ PDF criado: {CAMINHO_PDF}")

In [ ]:
# ─── 2. Converter com Docling
converter = DocumentConverter()
resultado = converter.convert(CAMINHO_PDF)
doc = resultado.document
markdown = doc.export_to_markdown()

print("✅ Markdown extraído:")
print(markdown)

In [ ]:
# ─── 3. Gerar chunks
chunker = HybridChunker(tokenizer='BAAI/bge-m3', max_tokens=400, merge_peers=True)
chunks = list(chunker.chunk(doc))

print(f"✅ {len(chunks)} chunk(s) gerado(s):")
for i, c in enumerate(chunks):
    print(f"\nChunk {i+1} ({len(c.text)} chars):")
    print(c.text)

In [ ]:
# ─── 4. Adicionar metadados e preparar para indexação
docs_para_indexar = []
for i, chunk in enumerate(chunks):
    docs_para_indexar.append({
        'id': f'exemplo1_chunk_{i}',
        'texto': chunk.text,
        'arquivo': 'exemplo_docling.pdf',
        'tipo': 'sumula',
        'tribunal': 'STJ',
        'chunk_index': i
    })

print("✅ Documentos prontos para indexação:")
print(json.dumps(docs_para_indexar, ensure_ascii=False, indent=2))

## 💡 Checklist de Uso

Use este template sempre que precisar de ingestão rápida:

- [ ] PDF disponível em disco (ou URL)
- [ ] `DocumentConverter()` para PDFs nativos
- [ ] `DocumentConverter` com `PdfFormatOption(do_ocr=True)` para escaneados
- [ ] `HybridChunker(tokenizer='BAAI/bge-m3', max_tokens=400)` para chunking
- [ ] Adicionar metadados: `arquivo`, `tipo`, `tribunal`, `data`
- [ ] Gerar embeddings com `SentenceTransformer('BAAI/bge-m3')`
- [ ] Indexar no OpenSearch ou FAISS
